# Thực nghiệm So sánh PhoBERT, viBERT, XLM-RoBERTa và mBERT cho Bài toán Phân loại Văn bản Tiếng Việt

Notebook này thực hiện quy trình thực nghiệm chuẩn (end-to-end) để so sánh hiệu năng, thời gian huấn luyện và tốc độ suy luận của 4 mô hình Pre-trained Language Models phổ biến trên bộ dữ liệu phân tích cảm xúc tiếng Việt **UIT-VSFC** (Vietnamese Students' Feedback Corpus):
1. **`vinai/phobert-base-v2`** (hoặc `vinai/phobert-base`) — Đơn ngữ tiếng Việt, có bước tách từ (`pyvi` / `underthesea` / `vncorenlp`).
2. **`FPTAI/vibert-base-cased`** — Đơn ngữ tiếng Việt, kiến trúc BERT gốc.
3. **`xlm-roberta-base`** — Đa ngôn ngữ kiến trúc RoBERTa.
4. **`bert-base-multilingual-cased`** (**mBERT**) — Đa ngôn ngữ kiến trúc BERT gốc.

> **Khuyến nghị môi trường:** Chạy trên Google Colab với GPU (T4 / L4 / A100) hoặc máy cục bộ có CUDA.

## 1. Cài đặt thư viện cần thiết

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pyvi matplotlib seaborn pandas

## 2. Import thư viện & Cấu hình chung (Seed, Device, Hyperparameters)

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from pyvi import ViTokenizer
from datasets import load_dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

# Tắt cảnh báo tokenizer parallelism
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Cố định seed để đảm bảo tính tái lập (reproducibility)
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị đang sử dụng: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Tải và Khám phá dữ liệu (UIT-VSFC)
Bộ dữ liệu **UIT-VSFC** gồm ~16.000 câu phản hồi của sinh viên tiếng Việt được gán nhãn 3 lớp cảm xúc:
- `0`: Tiêu cực (Negative)
- `1`: Trung tính (Neutral)
- `2`: Tích cực (Positive)

*(Nếu bạn muốn dùng bộ dữ liệu CSV riêng của mình, chỉ cần thay thế phần load dữ liệu bằng `pd.read_csv(...)` với 2 cột `text` và `label`).*

In [ ]:
from datasets import load_dataset, DatasetDict

# Tải UIT-VSFC từ nhánh refs/convert/parquet chính thức của Hugging Face Hub
# (Tương thích 100% với datasets >= 4.0 mà không cần script .py)
base_parquet_url = "https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/refs%2Fconvert%2Fparquet/default"

raw_datasets = load_dataset(
    "parquet",
    data_files={
        "train": f"{base_parquet_url}/train/0000.parquet",
        "validation": f"{base_parquet_url}/validation/0000.parquet",
        "test": f"{base_parquet_url}/test/0000.parquet",
    }
)

dataset = DatasetDict({
    split: raw_datasets[split].rename_columns({"sentence": "text", "sentiment": "label"})
    for split in ["train", "validation", "test"]
})

print(dataset)

LABEL_NAMES = ["Negative (0)", "Neutral (1)", "Positive (2)"]
NUM_LABELS = 3

# Xem thử 5 mẫu dữ liệu đầu tiên
df_sample = pd.DataFrame(dataset["train"][:5])
df_sample["label_name"] = df_sample["label"].map(lambda x: LABEL_NAMES[x])
display(df_sample[["text", "label", "label_name"]])

In [ ]:
# Kiểm tra phân bố nhãn trên tập Train / Validation / Test
dist_data = []
for split in ["train", "validation", "test"]:
    counts = pd.Series(dataset[split]["label"]).value_counts().sort_index()
    for lbl_idx, cnt in counts.items():
        dist_data.append({
            "Split": split,
            "Label": LABEL_NAMES[lbl_idx],
            "Count": cnt
        })

df_dist = pd.DataFrame(dist_data)
plt.figure(figsize=(8, 4))
sns.barplot(data=df_dist, x="Label", y="Count", hue="Split")
plt.title("Phân bố nhãn cảm xúc trên bộ dữ liệu UIT-VSFC")
plt.tight_layout()
plt.show()

## 4. Tiền xử lý đặc thù cho từng mô hình
- **PhoBERT** yêu cầu văn bản tiếng Việt phải được **tách từ** trước (ví dụ: `"giảng viên dạy rất nhiệt tình"` $\rightarrow$ `"giảng_viên dạy rất nhiệt_tình"`). Ở đây ta dùng `ViTokenizer.tokenize` từ `pyvi` (nhẹ, không cần Java như VnCoreNLP).
- **viBERT, XLM-RoBERTa, mBERT** sử dụng trực tiếp văn bản thô qua subword tokenizer tương ứng.

In [ ]:
def segment_vietnamese_text(example):
    example["text_segmented"] = ViTokenizer.tokenize(str(example["text"]))
    return example

print("Đang tách từ tiếng Việt cho PhoBERT...")
dataset = dataset.map(segment_vietnamese_text)

print("Ví dụ gốc      :", dataset["train"][0]["text"])
print("Ví dụ tách từ  :", dataset["train"][0]["text_segmented"])

## 5. Cấu hình danh sách 4 mô hình và Hàm huấn luyện / đánh giá chuẩn hóa
Để phép so sánh công bằng, cả 4 mô hình đều dùng chung:
- Độ dài chuỗi tối đa `MAX_LEN = 128`
- Số epoch: `EPOCHS = 4`
- Learning rate: `2e-5`, Weight decay: `0.01`, Warmup ratio: `0.1`
- Batch size: `32`
- Chọn checkpoint tốt nhất dựa trên `eval_f1_macro`

In [ ]:
MODELS_CONFIG = [
    {
        "name": "PhoBERT-base",
        "model_id": "vinai/phobert-base-v2",
        "use_word_segment": True
    },
    {
        "name": "viBERT-base",
        "model_id": "FPTAI/vibert-base-cased",
        "use_word_segment": False
    },
    {
        "name": "XLM-RoBERTa-base",
        "model_id": "xlm-roberta-base",
        "use_word_segment": False
    },
    {
        "name": "mBERT-base",
        "model_id": "bert-base-multilingual-cased",
        "use_word_segment": False
    }
]

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 4
LEARNING_RATE = 2e-5

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    _, _, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {
        "accuracy": acc,
        "precision_macro": prec_macro,
        "recall_macro": rec_macro,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

In [ ]:
import inspect

def run_experiment(config, dataset_dict):
    model_name = config["name"]
    model_id = config["model_id"]
    text_col = "text_segmented" if config["use_word_segment"] else "text"

    print("=" * 70)
    print(f"BẮT ĐẦU THỰC NGHIỆM: {model_name} ({model_id})")
    print(f"Cột văn bản đầu vào: {text_col}")
    print("=" * 70)

    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    def tokenize_fn(batch):
        return tokenizer(
            batch[text_col],
            truncation=True,
            max_length=MAX_LEN
        )

    cols_to_remove = [c for c in ["text", "text_segmented", "topic"] if c in dataset_dict["train"].column_names]
    tokenized_ds = dataset_dict.map(
        tokenize_fn,
        batched=True,
        remove_columns=cols_to_remove
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=NUM_LABELS
    )
    total_params_m = sum(p.numel() for p in model.parameters()) / 1e6

    output_dir = f"./results_{model_name.lower().replace('-', '_')}"
    total_steps = max(1, (len(tokenized_ds["train"]) // BATCH_SIZE) * EPOCHS)
    warmup_steps = int(0.1 * total_steps)

    # Tự động tương thích mọi phiên bản Transformers (4.x và 5.x)
    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    ta_kwargs = dict(
        output_dir=output_dir,
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        seed=SEED
    )
    if "eval_strategy" in ta_params:
        ta_kwargs["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in ta_params:
        ta_kwargs["evaluation_strategy"] = "epoch"

    training_args = TrainingArguments(**ta_kwargs)

    trainer_params = inspect.signature(Trainer.__init__).parameters
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["validation"],
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics
    )
    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    # 1. Đo thời gian huấn luyện
    t0 = time.time()
    trainer.train()
    train_time_sec = time.time() - t0

    # 2. Đánh giá trên tập Test & đo thời gian suy luận
    t1 = time.time()
    pred_output = trainer.predict(tokenized_ds["test"])
    infer_time_sec = time.time() - t1

    y_true = pred_output.label_ids
    y_pred = np.argmax(pred_output.predictions, axis=-1)
    metrics = pred_output.metrics

    print(f"\n--- Classification Report ({model_name} - Test Set) ---")
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

    # Giải phóng VRAM sau mỗi mô hình
    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "Model": model_name,
        "Checkpoint": model_id,
        "Params (M)": round(total_params_m, 1),
        "Accuracy (%)": round(metrics["test_accuracy"] * 100, 2),
        "Macro-F1 (%)": round(metrics["test_f1_macro"] * 100, 2),
        "Weighted-F1 (%)": round(metrics["test_f1_weighted"] * 100, 2),
        "Macro-Precision (%)": round(metrics["test_precision_macro"] * 100, 2),
        "Macro-Recall (%)": round(metrics["test_recall_macro"] * 100, 2),
        "Train Time (s)": round(train_time_sec, 1),
        "Infer Time (s)": round(infer_time_sec, 2),
        "y_true": y_true,
        "y_pred": y_pred
    }

## 6. Chạy thực nghiệm lần lượt cả 4 mô hình

In [ ]:
all_results = []

for cfg in MODELS_CONFIG:
    res = run_experiment(cfg, dataset)
    all_results.append(res)

## 7. Bảng tổng hợp kết quả & Trực quan hóa so sánh

In [ ]:
# Tạo bảng tổng hợp và sắp xếp theo Macro-F1 giảm dần
summary_cols = [
    "Model", "Params (M)", "Accuracy (%)", "Macro-F1 (%)",
    "Weighted-F1 (%)", "Macro-Precision (%)", "Macro-Recall (%)",
    "Train Time (s)", "Infer Time (s)"
]
df_summary = pd.DataFrame(all_results)[summary_cols].sort_values(by="Macro-F1 (%)", ascending=False)
df_summary.to_csv("vietnamese_text_classification_benchmark.csv", index=False)

print("=== BẢNG TỔNG HỢP KẾT QUẢ THỰC NGHIỆM TRÊN TẬP TEST ===")
display(df_summary)

In [ ]:
# Biểu đồ 1: So sánh Accuracy, Macro-F1 và Weighted-F1
df_melted = df_summary.melt(
    id_vars=["Model"],
    value_vars=["Accuracy (%)", "Macro-F1 (%)", "Weighted-F1 (%)"],
    var_name="Metric",
    value_name="Score"
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=df_melted, x="Model", y="Score", hue="Metric", ax=axes[0], palette="viridis")
axes[0].set_title("So sánh Độ chính xác & F1-Score trên tập Test")
axes[0].set_ylim(max(0, df_melted["Score"].min() - 10), 100)
axes[0].set_ylabel("Điểm số (%)")
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%.1f", fontsize=8, padding=2)

# Biểu đồ 2: So sánh Thời gian huấn luyện
sns.barplot(data=df_summary, x="Model", y="Train Time (s)", hue="Model", legend=False, ax=axes[1], palette="magma")
axes[1].set_title("So sánh Thời gian huấn luyện (giây)")
axes[1].set_ylabel("Thời gian (s)")
for container in axes[1].containers:
    axes[1].bar_label(container, fmt="%.1f", fontsize=9, padding=2)

plt.tight_layout()
plt.savefig("benchmark_comparison_charts.png", dpi=200)
plt.show()

In [ ]:
# Biểu đồ 3: Ma trận nhầm lẫn (Confusion Matrix) của cả 4 mô hình
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for idx, res in enumerate(all_results):
    cm = confusion_matrix(res["y_true"], res["y_pred"])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Neg", "Neu", "Pos"],
        yticklabels=["Neg", "Neu", "Pos"],
        cbar=False,
        ax=axes[idx]
    )
    axes[idx].set_title(f"{res['Model']}\nMacro-F1: {res['Macro-F1 (%)']}%")
    axes[idx].set_xlabel("Dự đoán")
    if idx == 0:
        axes[idx].set_ylabel("Thực tế")

plt.tight_layout()
plt.savefig("confusion_matrices_4_models.png", dpi=200)
plt.show()

## 8. Phân tích lỗi sai (Error Analysis) giữa PhoBERT và mBERT
Kiểm tra những câu mà **PhoBERT** dự đoán đúng nhưng **mBERT** dự đoán sai để thấy rõ tác động của tách từ tiếng Việt và mô hình đơn ngữ.

In [ ]:
res_map = {r["Model"]: r for r in all_results}
phobert_preds = res_map["PhoBERT-base"]["y_pred"]
mbert_preds = res_map["mBERT-base"]["y_pred"]
y_true = res_map["PhoBERT-base"]["y_true"]
test_texts = dataset["test"]["text"]

diff_cases = []
for i in range(len(y_true)):
    if phobert_preds[i] == y_true[i] and mbert_preds[i] != y_true[i]:
        diff_cases.append({
            "Văn bản": test_texts[i],
            "Nhãn thực tế": LABEL_NAMES[y_true[i]],
            "PhoBERT dự đoán": LABEL_NAMES[phobert_preds[i]],
            "mBERT dự đoán": LABEL_NAMES[mbert_preds[i]]
        })

print(f"Số mẫu PhoBERT đúng nhưng mBERT sai: {len(diff_cases)} / {len(y_true)}")
display(pd.DataFrame(diff_cases).head(10))